### Q1. First trace

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider


'''
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
)
'''

from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor
)

from sqlite_exporter import SQLiteSpanExporter

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from starter import rag
from starter import RAGBase

class RAGTraced(RAGBase):

    def search(self, query, num_results=5):

        with tracer.start_as_current_span(
            "search"
        ):

            print("search() called")

            return super().search(
                query,
                num_results
            )


    def llm(self, prompt):

        with tracer.start_as_current_span(
            "llm"
        ) as span:

            print("llm() called")

            response = super().llm(prompt)


            usage = response.usage


            print(
                "Input tokens:",
                usage.input_tokens
            )

            print(
                "Output tokens:",
                usage.output_tokens
            )

            span.set_attribute(
                "input_tokens",
                usage.input_tokens
            )

            span.set_attribute(
                "output_tokens",
                usage.output_tokens
            )

            input_cost = (
                usage.input_tokens / 1_000_000
            ) * 0.25


            output_cost = (
                usage.output_tokens / 1_000_000
            ) * 2

            span.set_attribute(
                "cost",
                input_cost + output_cost
            )

            return response


    def rag(self, query):

        with tracer.start_as_current_span(
            "rag"
        ):

            print("rag() called")

            search_results = self.search(query)

            prompt = self.build_prompt(
                query,
                search_results
            )

            response = self.llm(prompt)

            return response.output_text

traced_rag = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client
)


In [3]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)

print(answer)

rag() called
search() called
llm() called
Input tokens: 7111
Output tokens: 86
It keeps calling the model inside a `while True` loop and checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, it `break`s and stops.

So the exit condition is: **no function calls in the latest response**.


### Q2. Capturing metrics as span attributes

Answer: Input tokens: 7111

### Q3. Span timing

Answer: Over 2000ms

### Q4. Saving traces to SQLite

In [4]:
import sqlite3
import pandas as pd


conn = sqlite3.connect(
    "traces.db"
)


df = pd.read_sql(
    "SELECT * FROM spans",
    conn
)


print(df)

     name           start_time             end_time  input_tokens  \
0  search  1784738545382764400  1784738545387759700           NaN   
1     llm  1784738545393787000  1784738549749378500        7111.0   
2     rag  1784738545382764400  1784738549760749000           NaN   

   output_tokens     cost  
0            NaN      NaN  
1           86.0  0.00195  
2            NaN      NaN  


### Q5. Querying trace data

In [5]:
df = pd.read_sql(
    """
    SELECT
       name,
       SUM(
       (end_time-start_time)/1000000
       ) duration_ms
    FROM spans
    WHERE name!='rag'
    GROUP BY name
    """,
    conn
)


print(df)

     name  duration_ms
0     llm         4355
1  search            4


### Q6. Token stability across runs

In [6]:
for i in range(4):

    answer = rag.rag(
        "How does the agentic loop keep calling the model until it stops?"
    )

    print(answer)

The loop keeps calling the model by checking whether the model returned any `function_call` items.

- It sends the current `messages` to the model.
- If the response contains a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- After processing the response, if `has_function_calls` is still `False`, the loop breaks.

So the stop condition is: **no function calls in the latest response**.
It keeps calling the model in a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is: **no function calls this turn**.
It keeps a `while True` loop and checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tool, appends

In [9]:
df = pd.read_sql(
    """
    SELECT
        input_tokens
    FROM spans
    WHERE name='llm'
    """,
    conn
)


print(df)

   input_tokens
0          7111
